# Semana 30 — Synthetic Control
**Fase 6: Causal Inference e Experimentation** · Notebook de exercícios

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NetoRibeiro/ai-architect-foundations/blob/main/06-Phase-6-Causal-Inference/notebooks/semana_30_exercicio.ipynb)

Leitura de conceito desta semana: [Semana-30-Conceitos.md](https://github.com/NetoRibeiro/ai-architect-foundations/blob/main/06-Phase-6-Causal-Inference/docs/Semana-30-Conceitos.md)

Complete cada trecho marcado com `# [SEU CÓDIGO AQUI]`. As células seguintes dependem das anteriores: execute em ordem. O gabarito está em `semana_30_gabarito.ipynb`.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.optimize import minimize

## Passo 1 — O cenário
O modo "Porta-Retrato Digital" de um app de fotos é lançado **apenas na Espanha** na semana 60. Temos o engajamento semanal (minutos por usuário) de 16 países por 90 semanas. Efeito verdadeiro: cresce até **+6 minutos**.

In [ ]:
T, T0, EFEITO = 90, 60, 6.0
def gerar(seed=3):
    r = np.random.default_rng(seed)
    paises = ["Espanha", "Portugal", "Itália", "Brasil", "França", "México", "Argentina", "Chile", "Alemanha", "Reino Unido",
              "Colômbia", "Peru", "Holanda", "Bélgica", "Grécia", "Polônia"]
    t = np.arange(T); fatores = np.vstack([0.08 * t, 3 * np.sin(2 * np.pi * t / 26), np.cumsum(r.normal(0, 0.35, T))])   # tendência, sazonalidade, ciclo comum
    cargas = r.uniform(0.3, 1.7, (len(paises), 3)); niveis = r.uniform(20, 45, len(paises))
    cargas[0] = 0.5 * cargas[2] + 0.35 * cargas[1] + 0.15 * cargas[3]; niveis[0] = 0.5 * niveis[2] + 0.35 * niveis[1] + 0.15 * niveis[3]   # Espanha ≈ mistura
    Y = niveis[:, None] + cargas @ fatores + r.normal(0, 0.6, (len(paises), T))
    Y[0, T0:] += EFEITO * (1 - np.exp(-(t[T0:] - T0 + 1) / 6))                                                       # efeito que cresce e estabiliza
    return pd.DataFrame(Y.T, columns=paises)

df = gerar(); doadores = [c for c in df.columns if c != "Espanha"]
ax = df[doadores].plot(figsize=(10, 4), legend=False, color="lightgray"); df["Espanha"].plot(ax=ax, color="crimson", lw=2); plt.axvline(T0, ls="--", c="k"); plt.ylabel("minutos por usuário"); plt.show()

## Passo 2 — Tentativas ingênuas
(a) Antes/depois na Espanha. (b) DiD com um único país de controle, escolhido "por intuição" (França).

In [ ]:
pre, pos = df.iloc[:T0], df.iloc[T0:]
# [SEU CÓDIGO AQUI] (a) antes/depois; (b) DiD contra a França
efeito_medio_real = (EFEITO * (1 - np.exp(-(np.arange(T - T0) + 1) / 6))).mean()
print(f"antes/depois: {antes_depois:.2f} | DiD com França: {did_franca:.2f} | efeito médio verdadeiro: {efeito_medio_real:.2f}")

## Passo 3 — Os pesos do synthetic control
Minimize o erro quadrático entre a Espanha e a combinação dos donors **no pre-period**, com pesos não negativos que somam 1. Use `scipy.optimize.minimize` com `method="SLSQP"`.

In [ ]:
def pesos_sinteticos(y_pre, X_pre):
    """y_pre: (T0,), X_pre: (T0, J). Devolve w com w >= 0 e soma 1."""
    J = X_pre.shape[1]
    # [SEU CÓDIGO AQUI] defina a loss, resolva com SLSQP (bounds e restrição de soma 1) e devolva res.x

w = pesos_sinteticos(pre["Espanha"].values, pre[doadores].values)
pesos = pd.Series(w, index=doadores).sort_values(ascending=False); print(pesos[pesos > 0.01].round(3))
assert abs(w.sum() - 1) < 1e-6 and (w > -1e-8).all()

Quantos donors receberam peso relevante? A simulação construiu a Espanha como 50% Itália + 35% Portugal + 15% Brasil. Se os pesos encontrados forem outros, não é erro: com muitos donors correlacionados, várias combinações reproduzem igualmente bem o pre-period. Os pesos não são únicos; o que se avalia é a qualidade do ajuste.

## Passo 4 — Espanha real × Espanha sintética

In [ ]:
# [SEU CÓDIGO AQUI] série do synthetic control para todo o período
plt.figure(figsize=(10, 3.5)); plt.plot(df["Espanha"].values, c="crimson", label="Espanha"); plt.plot(sintetico, c="k", ls="--", label="Espanha sintética")
plt.axvline(T0, ls=":", c="gray"); plt.legend(); plt.ylabel("minutos por usuário"); plt.show()

## Passo 5 — Qualidade do ajuste e efeito
RMSPE no pre-period e efeito (diferença) no post-period.

In [ ]:
gap = df["Espanha"].values - sintetico
# [SEU CÓDIGO AQUI] rmspe_pre e efeito médio no post-period
print(f"RMSPE pré: {rmspe_pre:.2f} | efeito médio estimado: {efeito_medio:.2f} | verdadeiro: {efeito_medio_real:.2f} | efeito nas últimas 10 semanas: {gap[-10:].mean():.2f} (verdade ≈ {EFEITO})")
plt.figure(figsize=(10, 3)); plt.plot(gap, c="crimson"); plt.axhline(0, c="k", lw=.8); plt.axvline(T0, ls=":", c="gray"); plt.ylabel("Espanha − sintética"); plt.show()

## Passo 6 — In-space placebo
Aplique o mesmo procedimento a cada donor, fingindo que ele foi o tratado (os demais donors formam o seu grupo de controle; a Espanha fica de fora).

In [ ]:
def gap_de(unidade):
    # [SEU CÓDIGO AQUI] pesos para `unidade` usando os outros donors; devolva a série de diferenças

gaps = {"Espanha": gap, **{c: gap_de(c) for c in doadores}}
plt.figure(figsize=(10, 3.5))
for c, g in gaps.items(): plt.plot(g, c="crimson" if c == "Espanha" else "lightgray", lw=2 if c == "Espanha" else 1)
plt.axhline(0, c="k", lw=.8); plt.axvline(T0, ls=":", c="gray"); plt.ylabel("unidade − sintética"); plt.show()

## Passo 7 — p-value por permutação
Para cada unidade, calcule `RMSPE pós / RMSPE pré`. O p-value é a fração de unidades com razão maior ou igual à da Espanha.

In [ ]:
def razao(g):
    # [SEU CÓDIGO AQUI] RMSPE pós / RMSPE pré

razoes = pd.Series({c: razao(g) for c, g in gaps.items()}).sort_values(ascending=False)
# [SEU CÓDIGO AQUI] fração das unidades com razão >= à da Espanha
print(razoes.head(5).round(2)); print(f"\np-valor placebo: {p_valor:.3f} (mínimo possível com {len(razoes)} unidades: {1 / len(razoes):.3f})")

## Passo 8 — In-time placebo
Finja que o lançamento foi na semana 40 e use só as semanas 0 a 59. O "efeito" entre as semanas 40 e 59 deve ser próximo de zero.

In [ ]:
T_falso = 40
# [SEU CÓDIGO AQUI] pesos com dados até T_falso; diferença entre T_falso e T0
print(f"efeito placebo (semanas 40-59): {gap_f.mean():.2f} | RMSPE pré: {rmspe_pre:.2f}")

## Passo 9 — Doador contaminado
Suponha que a campanha vazou para o país de **maior peso** no sintético, que ganhou +3 minutos após o lançamento. Refaça a estimativa com esse país entre os donors e depois sem ele.

In [ ]:
contaminado = pesos.index[0]; df_c = df.copy(); df_c.loc[T0:, contaminado] += 3.0
def estimar(dados, lista):
    # [SEU CÓDIGO AQUI] efeito médio no post-period usando os donors em `lista`

print(f"doador contaminado: {contaminado} (peso {pesos.iloc[0]:.2f})")
print(f"com ele: {estimar(df_c, doadores):.2f} | sem ele: {estimar(df_c, [c for c in doadores if c != contaminado]):.2f} | verdadeiro: {efeito_medio_real:.2f}")

O viés é aproximadamente `peso × vazamento`. Excluir o donor contaminado custa pouco quando há outros donors bons.

## Fechamento da Fase 6
Monte o relatório pedido no entregável. Feche com um quadro: para cada método desta fase (A/B, A/B por cluster, DiD, PSM, IV, synthetic control), quando usar e qual a hipótese que não se pode testar.